# 머신러닝 원자간 포텐셜 실습

**Machine Learning Interatomic Potential · MLIP · MLFF · 기계학습 포텐셜**

DFT 등의 에너지·힘 데이터를 학습해 원자 배열에서 에너지와 힘을 빠르게 예측하는 모델. 대규모·장시간 시뮬레이션을 가능하게 하되 학습 범위를 벗어나면 신뢰도가 떨어진다.

소재 분야에서 이해하기: 학습된 포텐셜로 수천 원자 계의 확산을 계산한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [MACE 원자간 포텐셜 논문](https://arxiv.org/abs/2206.07697)

## 1. 비싼 계산을 흉내내는 참조 포텐셜

실제로는 DFT가 참조입니다. 여기서는 레너드-존스를 "비싼 참조"로 두고,
그 에너지와 힘을 학습한 포텐셜을 만들어 실제로 MD를 돌려 비교합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def lj_energy_forces(positions, box, cutoff=2.5):
    delta = positions[:, None, :] - positions[None, :, :]
    delta -= box * np.round(delta / box)
    distance2 = (delta ** 2).sum(-1)
    np.fill_diagonal(distance2, np.inf)
    mask = distance2 < cutoff ** 2
    inv6 = np.where(mask, (1.0 / distance2) ** 3, 0.0)
    energy = 4 * (inv6 ** 2 - inv6)[mask].sum() / 2
    magnitude = np.where(mask, 24 * (2 * inv6 ** 2 - inv6) / distance2, 0.0)
    return energy, (magnitude[:, :, None] * delta).sum(1)

def descriptors(positions, box, cutoff=2.5, widths=(0.4, 0.8, 1.6)):
    """원자별 방사 대칭함수: 이웃 거리를 가우시안으로 합산한 값."""
    delta = positions[:, None, :] - positions[None, :, :]
    delta -= box * np.round(delta / box)
    distance = np.sqrt((delta ** 2).sum(-1))
    np.fill_diagonal(distance, np.inf)
    mask = distance < cutoff
    features = []
    for centre in (1.0, 1.3, 1.6, 2.0):
        for width in widths:
            contribution = np.where(mask, np.exp(-((distance - centre) ** 2) / (2 * width ** 2)), 0.0)
            features.append(contribution.sum(1))
    return np.column_stack(features)

box = 6.0
print('참조 포텐셜과 기술자 준비 완료. 기술자 차원 %d' % descriptors(rng.random((10, 2)) * box, box).shape[1])

## 2. 참조 데이터 모으기

무작위 배열을 만들어 참조 에너지와 힘을 계산합니다(실제로는 이 단계가 비쌉니다).

In [ ]:
def perturbed_lattice(spacing, amplitude, local):
    """격자를 흔들어 물리적으로 가능한 배열을 만듭니다(무작위 배치는 원자가 겹쳐 발산합니다)."""
    base = np.array([[i * spacing, j * spacing] for i in range(5) for j in range(5)])
    return base + local.normal(0, amplitude, base.shape), 5 * spacing

features, energies, rejected = [], [], 0
for sample in range(600):
    local = np.random.default_rng(sample)
    positions, sample_box = perturbed_lattice(local.uniform(1.05, 1.45), local.uniform(0.01, 0.16), local)
    delta = positions[:, None, :] - positions[None, :, :]
    delta -= sample_box * np.round(delta / sample_box)
    distance = np.sqrt((delta ** 2).sum(-1)); np.fill_diagonal(distance, np.inf)
    if distance.min() < 0.85:          # 참조 계산이 발산하는 배열은 제외
        rejected += 1
        continue
    energy, _ = lj_energy_forces(positions, sample_box)
    features.append(descriptors(positions, sample_box).sum(0))   # 구조 기술자 = 원자 기술자 합
    energies.append(energy)
features, energies = np.array(features), np.array(energies)
print('학습 구조 %d개 (제외 %d개), 에너지 범위 %.1f ~ %.1f'
      % (len(energies), rejected, energies.min(), energies.max()))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

X_tr, X_te, y_tr, y_te = train_test_split(features, energies, test_size=0.25, random_state=0)
model = make_pipeline(StandardScaler(), KernelRidge(alpha=1e-3, kernel='rbf', gamma=0.02)).fit(X_tr, y_tr)
prediction = model.predict(X_te)
print('에너지 예측 MAE %.3f (에너지 표준편차 %.3f)' % (mean_absolute_error(y_te, prediction), energies.std()))
plt.scatter(y_te, prediction, s=14)
lo, hi = y_te.min(), y_te.max(); plt.plot([lo, hi], [lo, hi], 'k--')
plt.xlabel('reference energy'); plt.ylabel('MLIP energy'); plt.show()

## 3. 학습된 에너지에서 힘을 얻어 참조와 비교

힘은 학습된 에너지의 수치 미분으로 얻습니다. 대칭이 완벽한 격자는 힘이 0으로 상쇄되므로,
원자를 흔든 배열에서 비교합니다.

In [ ]:
def mlip_forces(positions, sample_box, step=2e-3):
    force = np.zeros_like(positions)
    for atom in range(len(positions)):
        for axis in range(2):
            shifted = positions.copy(); shifted[atom, axis] += step
            plus = model.predict(descriptors(shifted, sample_box).sum(0)[None, :])[0]
            shifted[atom, axis] -= 2 * step
            minus = model.predict(descriptors(shifted, sample_box).sum(0)[None, :])[0]
            force[atom, axis] = -(plus - minus) / (2 * step)
    return force

test_positions, test_box = perturbed_lattice(1.2, 0.12, np.random.default_rng(9999))
reference_force = lj_energy_forces(test_positions, test_box)[1]
learned_force = mlip_forces(test_positions, test_box)
print('참조 힘 크기 평균 %.3f / 학습 포텐셜 힘 크기 평균 %.3f'
      % (np.linalg.norm(reference_force, axis=1).mean(), np.linalg.norm(learned_force, axis=1).mean()))
correlation = np.corrcoef(reference_force.ravel(), learned_force.ravel())[0, 1]
print('힘 성분 상관계수 %.3f' % correlation)
plt.scatter(reference_force.ravel(), learned_force.ravel(), s=14)
limit = np.abs(reference_force).max() * 1.1
plt.plot([-limit, limit], [-limit, limit], 'k--', lw=1)
plt.xlabel('reference force component'); plt.ylabel('MLIP force component'); plt.show()
print('\n에너지만 학습했기 때문에 힘의 정확도는 에너지보다 떨어집니다.')
print('실제 MLIP 는 힘도 함께 손실에 넣어 학습합니다.')

## 4. 학습 범위를 벗어나면

원자 수는 그대로 두고 격자 간격만 줄여, 학습에 없던 짧은 거리 영역으로 밀어봅니다.

In [ ]:
print('학습에 쓰인 격자 간격 범위: 1.05 ~ 1.45')
for spacing in (1.40, 1.20, 1.00, 0.95, 0.90):
    positions, sample_box = perturbed_lattice(spacing, 0.02, np.random.default_rng(7))
    reference = lj_energy_forces(positions, sample_box)[0]
    learned = model.predict(descriptors(positions, sample_box).sum(0)[None, :])[0]
    tag = '학습 범위' if 1.05 <= spacing <= 1.45 else '범위 밖'
    print('간격 %.2f -> 참조 %10.2f / 학습 포텐셜 %10.2f   오차 %8.2f  (%s)'
          % (spacing, reference, learned, learned - reference, tag))
print('\n짧은 거리에서 참조는 급격히 발산하지만 학습 포텐셜은 그 거동을 배우지 못했습니다.')
print('실제 MLIP 운용에서도 이 "학습 영역 밖" 배열이 시뮬레이션 폭주의 주된 원인입니다.')
print('그래서 불확실성 추정과 능동학습으로 새 배열을 참조 계산에 보내 학습을 보강합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#mlip)을 여세요.